[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Link and BackLink


## What you will be able to do

Refer to one document from another with `Link`, and back the other way with `BackLink`. Say what a
`Link` actually is before it has been fetched, which is not the document and not an error either.
Fetch them: one at a time, which costs a round trip each, or with `fetch_links=True`, which is one
`$lookup` for the lot. Say what `WriteRules` decides when you save a document whose link points at
something that was never inserted. And know the nesting depth past which links come back
unresolved with no warning at all.


## The idea

### The problem

**Modeling Without Joins** established when to reference rather than embed. Doing it by hand means
storing an id, remembering which collection it belongs to, and writing the second query yourself.

`Link` is Beanie's version of that, with the type of the other model attached, so the second query
can be written for you. The catch is that it is not written unless you ask.

### What a Link is

A field holding a DBRef, which is an id and a collection name. Read a document normally and the
field is a `Link` object: not the target, not `None`, and perfectly happy to be passed around until
something asks it for a field it does not have.

### Why fetch_links exists

Fetching links one document at a time is one round trip per document, which is the classic
N plus 1. `fetch_links=True` puts `$lookup` stages into the query instead, so the whole set comes
back resolved in one trip.

### Where this shows up

Any model with a reference in it. The `AttributeError` below is the first thing that happens to
everybody, and the loop that fixes it is the second thing, and it is slower than the thing it
replaced.

### What this notebook covers

`Link` and what it holds. `fetch_link` for one, `fetch_links=True` for a query. `BackLink`, for the
other direction. `WriteRules`, and the link pointing at an uninserted document. Then the four
failures, one of which is silent and depends on how deep the links go.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import asyncio

from beanie import Document, Link, init_beanie
from pymongo import AsyncMongoClient


class Maker(Document):
    name: str

    class Settings:
        name = "makers"


class Product(Document):
    sku: str
    maker: Link[Maker]                             # a reference, not the document

    class Settings:
        name = "linked"


async def main():
    client = AsyncMongoClient("mongodb://127.0.0.1:27017/shop")
    await init_beanie(database=client.get_default_database(),
                      document_models=[Maker, Product])

    await Maker.delete_all()
    await Product.delete_all()
    maker = Maker(name="Aster")
    await maker.insert()
    await Product(sku="P-1", maker=maker).insert()

    loose = await Product.find_one(Product.sku == "P-1")
    print("read normally, maker is a:", type(loose.maker).__name__)
    try:
        print(loose.maker.name)
    except AttributeError as error:
        print("  and asking for its name:", error)

    full = await Product.find_one(Product.sku == "P-1", fetch_links=True)
    print("with fetch_links=True:    ", type(full.maker).__name__, "|", full.maker.name)
    await client.close()


asyncio.run(main())
```

```
read normally, maker is a: Link
  and asking for its name: 'Link' object has no attribute 'name'
with fetch_links=True:     Maker | Aster
```

The same field, read twice. Without `fetch_links` it is a `Link`, which holds an id and a collection
name and nothing else. With it, it is the `Maker`. Nothing in the model declaration says which you
will get; the query does.


## Setup

Thirteen imports, MongoDB, and the boot cell.

- `beanie` with `Link`, `BackLink`, `WriteRules`, `Document` and `init_beanie`
- `pydantic`'s `Field` is needed once, to tell a `BackLink` which field points back at it
- `typing` supplies `List` and `Optional`, which a `BackLink` field is declared with
- `warnings` silences one deprecation notice, explained where it is used
- `subprocess`, `os`, `sys`, `time`, `random`, `version` and `PackageNotFoundError` run the boot cell

There are no helpers: this notebook is about what the model declarations do.


In [1]:
import os
import random
import subprocess
import sys
import time
import warnings
from typing import List, Optional
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import beanie
import pymongo
from beanie import BackLink, Document, Link, WriteRules, init_beanie
from pydantic import Field
from pymongo import AsyncMongoClient

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### Two models, one pointing at the other


In [2]:
class Maker(Document):
    name: str
    country: str = "unknown"

    class Settings:
        name = "makers"


class Product(Document):
    sku: str
    price: float
    maker: Link[Maker]

    class Settings:
        name = "linked"


client = AsyncMongoClient(URI)
await init_beanie(database=client.get_default_database(), document_models=[Maker, Product])

await Maker.delete_all()
await Product.delete_all()

aster = Maker(name="Aster", country="IE")
belden = Maker(name="Belden", country="PT")
await aster.insert()                                                # one at a time, because
await belden.insert()                                               # insert_many leaves id unset

await Product.insert_many([
    Product(sku="P-1", price=10.0, maker=aster),
    Product(sku="P-2", price=20.0, maker=aster),
    Product(sku="P-3", price=30.0, maker=belden),
])
print("products:", await Product.find_all().count(), "| makers:", await Maker.find_all().count())


products: 3 | makers: 2


What is actually stored is a DBRef, which you can see by reading the collection without the model:


In [3]:
raw = await client.get_default_database().linked.find_one({"sku": "P-1"})
print("the stored field:", sorted(raw["maker"].keys()) if hasattr(raw["maker"], "keys")
      else type(raw["maker"]).__name__)
print("it names a collection and an id, and nothing else")


the stored field: DBRef
it names a collection and an id, and nothing else


### Fetching one

`fetch_link` resolves a single field in place:


In [4]:
product = await Product.find_one(Product.sku == "P-1")
print("before:", type(product.maker).__name__)

await product.fetch_link(Product.maker)
print("after: ", type(product.maker).__name__, "|", product.maker.name, product.maker.country)


before: Link
after:  Maker | Aster IE


`fetch_all_links()` does every link on the document at once. Both are one round trip per document,
which is fine for one document and is the problem for a list of them.

### Fetching a whole query

`fetch_links=True` builds the join into the query:


In [5]:
loose = await Product.find_all().to_list()
print("without:", [(p.sku, type(p.maker).__name__) for p in loose])

joined = await Product.find_all(fetch_links=True).to_list()
print("with:   ", [(p.sku, p.maker.name) for p in joined])


without: [('P-1', 'Link'), ('P-2', 'Link'), ('P-3', 'Link')]
with:    [('P-1', 'Aster'), ('P-2', 'Aster'), ('P-3', 'Belden')]


The difference is not only convenience. Without it, resolving three products means three more
queries; with it, the server does one aggregation with a `$lookup` in it and sends everything back
together.

This is the round trip per document that `fetch_links` **avoids**, and it is worth being precise
about the direction, because the feature is often described the other way round.

### Filtering on a linked field

With `fetch_links=True` the link's fields are available to the filter as well:


In [6]:
irish = await Product.find(Product.maker.country == "IE", fetch_links=True).to_list()
print("made in IE:", [product.sku for product in irish])

print("and without fetch_links the same filter finds:",
      len(await Product.find(Product.maker.country == "IE").to_list()))


made in IE: ['P-1', 'P-2']
and without fetch_links the same filter finds: 0


Zero, because without the join there is no `country` field on the product to filter on: there is a
DBRef. The filter is not wrong, it simply has nothing to match, which is the same class of quiet
failure as a misspelled field in **Query Operators**.

### BackLink, for the other direction

A `Link` points one way. To go the other way you declare a `BackLink` and say which field points
back at you:


In [7]:
with warnings.catch_warnings():                                     # see the note below
    warnings.simplefilter("ignore")

    class MakerWithProducts(Document):
        name: str
        country: str = "unknown"
        products: Optional[List[BackLink["Product"]]] = Field(default=None,
                                                              original_field="maker")

        class Settings:
            name = "makers"


await init_beanie(database=client.get_default_database(),
                  document_models=[Maker, Product, MakerWithProducts])

maker = await MakerWithProducts.find_one(MakerWithProducts.name == "Aster", fetch_links=True)
print("Aster's products:", sorted(product.sku for product in maker.products))


Aster's products: ['P-1', 'P-2']


`original_field="maker"` names the field on `Product` that points here. Nothing is stored on the
maker: the backlink is computed by looking for products whose `maker` is this one, which is why it
is only populated with `fetch_links=True`.

The `warnings.catch_warnings` is there for an honest reason: passing `original_field` to `Field` is
how Beanie's own documentation declares a backlink, and Pydantic 2 reports any unknown keyword to
`Field` as deprecated. The code is correct and the notice is noise, so it is silenced at the one
place it appears rather than everywhere.

### WriteRules

By default, saving a document does **not** save what its links point at:


In [8]:
await Maker.delete_all()
await Product.delete_all()

saved_first = Maker(name="Corvid", country="ES")
await saved_first.insert()
await Product(sku="Q-1", price=1.0, maker=saved_first).insert()
print("the ordinary way, maker exists:",
      await Maker.find_one(Maker.name == "Corvid") is not None)

await Product(sku="Q-2", price=2.0,
              maker=Maker(name="Dalgo", country="FR")).insert(link_rule=WriteRules.WRITE)
print("with WriteRules.WRITE:         ",
      await Maker.find_one(Maker.name == "Dalgo") is not None)


the ordinary way, maker exists: True
with WriteRules.WRITE:          True


`WriteRules.DO_NOTHING` is the default and it means exactly that: the link is written as a
reference and the target is left alone, which is right when the target is already in the database.
`WriteRules.WRITE` inserts the target too.

Getting this wrong does not produce a dangling reference, because Beanie refuses to write a
reference to a document with no id at all, which is the third error below.

### When to reach for which

| What you want | How to write it |
|---|---|
| a reference to another document | `field: Link[Other]` |
| the reverse direction | `Optional[List[BackLink["Other"]]]` with `original_field` |
| to resolve one field | `await document.fetch_link(Model.field)` |
| to resolve everything on one document | `await document.fetch_all_links()` |
| to resolve a whole query | `find(..., fetch_links=True)` |
| to filter on a linked document's field | `fetch_links=True`, and then you can |
| to save the target as well | `insert(link_rule=WriteRules.WRITE)` |
| the target left alone | nothing: that is the default |

The default is `fetch_links=True` on any query whose results will touch the linked documents, and
plain `find` when they will not. Resolving in a loop is the option to avoid, and it is the one that
looks most natural.

### A catalog that reads in one trip, finished


In [9]:
async def listing(country=None):
    """One query, links resolved by the server, and the filter allowed to reach into them."""
    query = Product.find(fetch_links=True)
    if country is not None:
        query = query.find(Product.maker.country == country)

    rows = await query.sort(Product.sku).to_list()
    return [{"sku": product.sku, "price": product.price,
             "maker": product.maker.name, "country": product.maker.country}
            for product in rows]


async def listing_the_slow_way():
    """The same answer, one extra round trip per product, which is the shape to recognize."""
    rows = await Product.find_all().sort(Product.sku).to_list()
    for product in rows:
        await product.fetch_link(Product.maker)                     # one query, each time round
    return [(product.sku, product.maker.name) for product in rows]


await Maker.delete_all()
await Product.delete_all()
makers = [Maker(name="Aster", country="IE"), Maker(name="Belden", country="PT")]
for maker in makers:
    await maker.insert()                                            # insert_many leaves id unset
await Product.insert_many([Product(sku=f"R-{n}", price=float(n), maker=makers[n % 2])
                           for n in range(4)])

print("everything:", await listing())
print()
print("just IE:   ", await listing(country="IE"))
print()
print("the slow way gives the same answer:", await listing_the_slow_way())


everything: [{'sku': 'R-0', 'price': 0.0, 'maker': 'Aster', 'country': 'IE'}, {'sku': 'R-1', 'price': 1.0, 'maker': 'Belden', 'country': 'PT'}, {'sku': 'R-2', 'price': 2.0, 'maker': 'Aster', 'country': 'IE'}, {'sku': 'R-3', 'price': 3.0, 'maker': 'Belden', 'country': 'PT'}]

just IE:    [{'sku': 'R-0', 'price': 0.0, 'maker': 'Aster', 'country': 'IE'}, {'sku': 'R-2', 'price': 2.0, 'maker': 'Aster', 'country': 'IE'}]

the slow way gives the same answer: [('R-0', 'Aster'), ('R-1', 'Belden'), ('R-2', 'Aster'), ('R-3', 'Belden')]


Both functions return the same thing. The first sends one query. The second sends one, then one more
for every row it got back, and the number of queries grows with the size of the result while the
code looks like an ordinary loop.

That is the whole argument for `fetch_links`, and the reason to reach for it by default on any query
whose results you are going to read through.

### Where each part came from

| In the listing | What it relies on | The section that showed it |
|---|---|---|
| `Link[Maker]` on the model | a field holding a DBRef | Two models, one pointing at the other |
| `find(fetch_links=True)` | `$lookup` built into the query | Fetching a whole query |
| `Product.maker.country == ...` | filtering through a resolved link | Filtering on a linked field |
| `fetch_link` in the slow version | one round trip per document | Fetching one |
| inserting the makers first | the default `WriteRules.DO_NOTHING` | WriteRules |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/14-link-and-backlink-solutions.ipynb).

**1.** Define two models, one linking to the other, and insert one of each.


In [10]:
# your code here


**2.** Read a document and print the type of its link field.


In [11]:
# your code here


**3.** Resolve that one link with `fetch_link`.


In [12]:
# your code here


**4.** Read a whole query with `fetch_links=True`.


In [13]:
# your code here


**5.** Filter on a field of the linked document.


In [14]:
# your code here


**6.** Insert a document whose link target has not been saved, two ways.


In [15]:
# your code here


## Common errors

### AttributeError: 'Link' object has no attribute


In [16]:
product = await Product.find_one(Product.sku == "R-0")
product.maker.name


AttributeError: 'Link' object has no attribute 'name'

The read succeeded, the document is fine, and `maker` is a `Link`: an id and a collection name. It
is not `None`, so a check for `None` will not catch this, and it is truthy, so `if product.maker:`
will not either.

Two fixes, and the first is the one that looks natural and is wrong at scale:


In [17]:
one = await Product.find_one(Product.sku == "R-0")
await one.fetch_link(Product.maker)
print("fetched one:", one.maker.name)

every = await Product.find_all(fetch_links=True).to_list()
print("fetched all:", [p.maker.name for p in every])


fetched one: Aster
fetched all: ['Aster', 'Belden', 'Aster', 'Belden']


### No error: the filter that matched nothing because the link was not fetched


In [18]:
without = await Product.find(Product.maker.country == "IE").to_list()
with_links = await Product.find(Product.maker.country == "IE", fetch_links=True).to_list()

print("without fetch_links:", len(without), "products")
print("with fetch_links:   ", len(with_links), "products")
print("the first query is legal, runs, and can never match anything")


without fetch_links: 0 products
with fetch_links:    2 products
the first query is legal, runs, and can never match anything


`maker.country` is not a field of a product. What is stored there is a DBRef, so the filter is
looking for a path that does not exist in the collection, and MongoDB answers honestly with nothing.

Whenever a filter reaches through a link, `fetch_links=True` is not optional. It is what creates the
field being filtered on.

### beanie.exceptions.DocumentWasNotSaved: Can not create dbref without id


In [19]:
never_saved = Maker(name="Ghost", country="XX")
await Product(sku="R-9", price=1.0, maker=never_saved).insert()


DocumentWasNotSaved: Can not create dbref without id

A `Link` is an id and a collection name, and this `Maker` has no id because it was never inserted.
Beanie refuses rather than writing a reference to nothing, which is better than the dangling
reference you would get from doing this by hand.

Insert the target first, or tell the insert to do it:


In [20]:
first = Maker(name="Ghost", country="XX")
await first.insert()
await Product(sku="R-9", price=1.0, maker=first).insert()
print("inserted after saving the maker:", await Product.find_one(Product.sku == "R-9") is not None)

await Product(sku="R-10", price=1.0,
              maker=Maker(name="Spectre", country="XX")).insert(link_rule=WriteRules.WRITE)
print("or with WriteRules.WRITE:       ",
      await Maker.find_one(Maker.name == "Spectre") is not None)


inserted after saving the maker: True
or with WriteRules.WRITE:        True


### No error: how far down fetch_links goes


In [21]:
class Region(Document):
    name: str

    class Settings:
        name = "regions"


class Supplier(Document):
    name: str
    region: Link[Region]

    class Settings:
        name = "suppliers"


class Part(Document):
    code: str
    supplier: Link[Supplier]

    class Settings:
        name = "parts"


await init_beanie(database=client.get_default_database(),
                  document_models=[Maker, Product, Region, Supplier, Part])
for model in (Region, Supplier, Part):
    await model.delete_all()

region = Region(name="North")
await region.insert()
supplier = Supplier(name="a supplier", region=region)
await supplier.insert()
await Part(code="X-1", supplier=supplier).insert()

part = await Part.find_one(Part.code == "X-1", fetch_links=True)
print("one level down:", type(part.supplier).__name__, "|", part.supplier.name)
print("two levels down:", type(part.supplier.region).__name__)


one level down: Supplier | a supplier
two levels down: Region


Two levels resolved, which is the useful half of the answer. The other half is that this does not
go on forever: Beanie stops at a maximum nesting depth, three by default, and a link past it comes
back as a `Link` while the ones above it came back as documents.

Nothing warns you when that happens. The same attribute access works at one depth and raises an
`AttributeError` one level further down, which is a difficult thing to see in a stack trace and an
easy thing to avoid: do not write code that depends on how deep the automatic fetching reaches.
Check, and fetch what you need where you need it:


In [22]:
if type(part.supplier.region).__name__ == "Link":
    await part.supplier.fetch_link(Supplier.region)

print("resolved either way:", type(part.supplier.region).__name__,
      "|", part.supplier.region.name)
print()
print("a chain of links this long is a sign the model wants flattening, not deeper fetching")


resolved either way: Region | North

a chain of links this long is a sign the model wants flattening, not deeper fetching


In [23]:
for model in (Maker, Product, Region, Supplier, Part):
    await model.delete_all()
await client.close()
print("tidied up and closed")


tidied up and closed


## Recap

- `Link[Other]` stores a DBRef: an id and a collection name. Read without fetching, the field is a
  `Link` object, which is truthy and is not the document.
- `'Link' object has no attribute ...` is what that turns into, and neither a `None` check nor a
  truthiness check will catch it first.
- `fetch_link` resolves one field and `fetch_all_links` resolves one document's worth, both at one
  round trip per document.
- `find(..., fetch_links=True)` builds `$lookup` stages into the query, so a whole result set comes
  back resolved in one trip. It is what **avoids** the round trip per document, not what causes it.
- A filter that reaches through a link needs `fetch_links=True` to have anything to match, and
  without it matches nothing and raises nothing.
- `BackLink` with `original_field` gives the reverse direction. Nothing is stored for it and it is
  populated only with `fetch_links=True`.
- Saving a document does not save what its links point at. `WriteRules.WRITE` changes that, and a
  link to a document with no id raises `DocumentWasNotSaved` rather than writing a dangling
  reference.
- `fetch_links` follows a chain of links, but only to a maximum nesting depth of three. Past it a
  link comes back unresolved, with no warning, so code should not depend on how far it reaches.


## What is next

**Migrations** is changing documents you have already written: `beanie new-migration`, the
iterative and free fall kinds, the models module a migration has to import, and the replica set the
command assumes you are running against.


---

&#8592; **Previous:** [Saving Changes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/13-saving-changes.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
